# Proyecto de aula · Corte 3 — Del dato a la decisión: un tablero en Python para una flota de transporte del Huila

**Curso:** Electiva VIII · Ciencia de Datos 2026-B · Corte 3  
**Docente:** Jesús Ariel González Bonilla  
**Institución:** Corporación Universitaria del Huila (CORHUILA) · Facultad de Ingeniería · Ingeniería Industrial  
**Formato institucional:** FO-IV-159 «Informe de proyectos de aula»  
**Herramientas:** Python (pandas, SQLite, matplotlib) en Google Colab  


## 0 · Overview & insights

**Problem.** Flota Alto Magdalena S.A.S., a fictitious Huila transport company with 60 vehicles, schedules maintenance without knowing which vehicle types and systems fail most often, so repairs are reactive and arrive late. **Data.** We generated three synthetic sources with Python (1,522 work-order records, 720 vehicle-month operating-hour records and 60 vehicles) covering November 2025 to October 2026 and seeded with deliberate quality problems. **Cleaning.** We removed 45 duplicated orders, standardised three date formats, normalised inconsistent text in vehicle IDs, system names and vehicle types, replaced 23 impossible repair times and filled 12 missing operating-hour values, which left 1,477 valid failures. **Analysis.** Using SQLite queries and pandas, we computed MTBF (operating hours divided by failures), MTTR and average cost per failure by month, vehicle type, system and shift. **Key insight.** Fleet MTBF fell 22.6%, from 171.7 hours in the first semester to 132.9 hours in the second; dump trucks are the weakest type at 119.5 hours, brake and tire failures on tractor-trailers and dump trucks rose from 112 to 204, and the night shift concentrates 43.2% of all failures. **Recommendation.** The workshop manager should schedule preventive brake and tire maintenance for tractor-trailers and dump trucks before May 2027 and add technician coverage on the night shift, so that the second-semester drop in reliability does not repeat.

## 1 · Ficha del proyecto

*En esta parte se copia la ficha de la Fase 0 (sección 4 de la guía) para que el lector conozca de entrada la empresa, la pregunta y los indicadores del proyecto.*

| Campo | Su respuesta |
|---|---|
| Integrantes (máximo dos) | Isabella Ramos Betancourt y José Julián Briñez Garzón |
| Empresa y tipo (real o ficticia) | Flota Alto Magdalena S.A.S. · **ficticia** (inventada para este proyecto; no corresponde a ninguna empresa real) |
| A qué se dedica | Empresa de transporte de carga y pasajeros del Huila con 60 vehículos (tractocamiones, camiones, volquetas, buses y camionetas) distribuidos en las sedes de Neiva, Pitalito y Garzón. Su taller central atiende las fallas correctivas de toda la flota, organizado en tres turnos (mañana, tarde y noche). |
| Problema | El taller programa el mantenimiento preventivo y asigna técnicos por turno con base en la experiencia del jefe de taller, sin medir cuántas horas operan los vehículos entre una falla y la siguiente. No sabe a tiempo qué tipos de vehículo y qué sistemas concentran las averías ni en qué meses empeoran, por lo que repara después de la falla en lugar de anticiparla. |
| Pregunta | ¿Qué tipo de vehículo y qué sistema explican la caída del tiempo medio entre fallas (MTBF) de la flota entre el primer semestre (nov. 2025–abr. 2026) y el segundo (may.–oct. 2026)? |
| KPI principal y fórmula | **MTBF** (tiempo medio entre fallas) = horas de operación totales ÷ número de fallas, calculado por mes, tipo de vehículo y sistema. |
| KPI de apoyo | **MTTR** (tiempo medio de reparación) = suma de horas de reparación ÷ número de fallas.<br>**Costo promedio por falla (COP)** = costo total de las órdenes con costo ÷ número de órdenes con costo. |
| Fuentes de datos y formato | Tres archivos CSV sintéticos generados con Python: `fallas.csv` (órdenes de trabajo correctivas), `horas_operacion.csv` (horas operadas por vehículo y mes) y `vehiculos.csv` (catálogo de la flota). |
| Autorización (solo empresa real) | No aplica: la empresa es ficticia. |

## 2 · Carga de datos

*Qué se hace y por qué:* la empresa es ficticia, así que primero se **generan** las tres fuentes con una función documentada y una semilla fija (resultados reproducibles), y luego se **leen** desde los archivos CSV como se haría con una exportación del software del taller. Se muestra el tamaño y una muestra de cada fuente.

**Cómo se generaron los datos y qué reglas se usaron para que fueran realistas** (todo se hizo con Python, `numpy` y `pandas`):

- La cantidad de fallas de cada vehículo en cada mes crece con sus **horas de operación**, con su **tipo** (las volquetas y los tractocamiones fallan más que las camionetas) y con su **antigüedad**.
- Los meses de **mayo a octubre** (segundo semestre del periodo) tienen más fallas por temporadas de mayor carga y de lluvias; diciembre y enero también presentan un pico.
- En ese segundo semestre, **tractocamiones y volquetas** aumentan las fallas de **frenos y llantas**.
- El **turno de noche** concentra más fallas y reparaciones más largas.
- Se inyectaron **problemas de calidad a propósito**: duplicados, fechas en tres formatos, textos con mayúsculas, tildes y espacios mezclados, valores imposibles (horas negativas o exageradas) y vacíos.

In [35]:
import os, re, sqlite3, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

**2.1 · Función generadora de datos sintéticos** (reglas descritas arriba; semilla `2026`).

In [36]:
import os
import unicodedata
import numpy as np
import pandas as pd


def generar_datos(carpeta="datos", semilla=2026):
    """Genera las tres fuentes sintéticas de Flota Alto Magdalena S.A.S. (empresa ficticia).

    Reglas de realismo (documentadas en la parte 2 del cuaderno):
      * Cada vehículo falla en proporción a sus horas de operación, a su tipo y a su antigüedad.
      * Los meses de mayo a octubre (segundo semestre del periodo) concentran más fallas
        por lluvias y por temporadas de mayor carga.
      * En ese semestre, tractocamiones y volquetas aumentan fallas de frenos y llantas.
      * El turno de noche concentra más fallas y reparaciones más largas.
    Luego se inyectan problemas de calidad a propósito (duplicados, fechas en formatos
    distintos, textos con mayúsculas/tildes mezcladas, valores imposibles y vacíos).
    """
    rng = np.random.default_rng(semilla)
    os.makedirs(carpeta, exist_ok=True)

    # ---------- 1. Vehículos ----------
    tipos = {
        "Tractocamión": (14, ["Kenworth", "International", "Freightliner"], 1.15, 360),
        "Camión": (16, ["Chevrolet", "Hino", "JAC"], 1.00, 290),
        "Volqueta": (12, ["Kenworth", "Chevrolet", "Hino"], 1.40, 300),
        "Bus": (8, ["Chevrolet", "Hino", "Mercedes-Benz"], 0.90, 330),
        "Camioneta": (10, ["Toyota", "Chevrolet", "Renault"], 0.60, 180),
    }
    filas_v = []
    n = 0
    for tipo, (cant, marcas, _, _) in tipos.items():
        for _ in range(cant):
            n += 1
            anio = int(rng.integers(2008, 2024))
            filas_v.append({
                "id_vehiculo": f"V{n:03d}",
                "tipo": tipo,
                "marca": str(rng.choice(marcas)),
                "anio": anio,
                "sede": str(rng.choice(["Neiva", "Pitalito", "Garzón"], p=[0.5, 0.3, 0.2])),
                "km_inicial": int((2026 - anio) * rng.integers(40000, 70000)),
            })
    veh = pd.DataFrame(filas_v)

    # ---------- 2. Horas de operación por vehículo y mes ----------
    meses = pd.period_range("2025-11", "2026-10", freq="M")
    h_mult = {11: 1.05, 12: 1.15, 1: 1.00, 2: 0.90, 3: 0.95, 4: 1.05,
              5: 1.10, 6: 1.10, 7: 1.00, 8: 1.00, 9: 1.00, 10: 1.10}
    f_mult = {11: 1.00, 12: 1.20, 1: 1.10, 2: 0.85, 3: 0.90, 4: 1.00,
              5: 1.30, 6: 1.35, 7: 1.20, 8: 1.15, 9: 1.25, 10: 1.40}
    filas_h = []
    factor_veh = {v: rng.uniform(0.85, 1.15) for v in veh["id_vehiculo"]}
    for _, v in veh.iterrows():
        base = tipos[v["tipo"]][3]
        for m in meses:
            h = base * h_mult[m.month] * factor_veh[v["id_vehiculo"]] * rng.normal(1, 0.08)
            filas_h.append({"id_vehiculo": v["id_vehiculo"], "mes": str(m),
                            "horas_operadas": float(round(max(h, 20), 1))})
    hor = pd.DataFrame(filas_h)

    # ---------- 3. Fallas (tabla principal) ----------
    sistemas = ["Motor", "Transmisión", "Frenos", "Eléctrico", "Suspensión", "Llantas", "Refrigeración"]
    p_base = np.array([0.20, 0.10, 0.18, 0.14, 0.14, 0.16, 0.08])
    h_rep = {"Motor": 14, "Transmisión": 18, "Frenos": 5, "Eléctrico": 4,
             "Suspensión": 6, "Llantas": 2, "Refrigeración": 7}
    c_rep = {"Motor": 2.4e6, "Transmisión": 3.1e6, "Frenos": 0.9e6, "Eléctrico": 0.6e6,
             "Suspensión": 1.1e6, "Llantas": 0.8e6, "Refrigeración": 1.0e6}
    f_tipo_rep = {"Tractocamión": 1.3, "Volqueta": 1.2}
    turnos = ["Mañana", "Tarde", "Noche"]
    p_turno = [0.28, 0.30, 0.42]
    km_v = dict(zip(veh["id_vehiculo"], veh["km_inicial"]))
    anio_v = dict(zip(veh["id_vehiculo"], veh["anio"]))
    tipo_v = dict(zip(veh["id_vehiculo"], veh["tipo"]))
    horas_idx = hor.set_index(["id_vehiculo", "mes"])["horas_operadas"]

    filas_f = []
    for (idv, mes), horas in horas_idx.items():
        per = pd.Period(mes)
        tipo = tipo_v[idv]
        edad = 2026 - anio_v[idv]
        lam = (horas / 250) * tipos[tipo][2] * (1 + 0.04 * edad) * f_mult[per.month]
        k = int(rng.poisson(lam))
        for _ in range(k):
            p = p_base.copy()
            if per.month in (5, 6, 7, 8, 9, 10) and tipo in ("Tractocamión", "Volqueta"):
                p[2] *= 1.8   # frenos
                p[5] *= 1.5   # llantas
            p = p / p.sum()
            sis = str(rng.choice(sistemas, p=p))
            turno = str(rng.choice(turnos, p=p_turno))
            dia = int(rng.integers(1, per.days_in_month + 1))
            fecha = pd.Timestamp(per.year, per.month, dia)
            horas_rep = (h_rep[sis] * f_tipo_rep.get(tipo, 1.0)
                         * (1.25 if turno == "Noche" else 1.0) * rng.lognormal(0, 0.45))
            costo = (c_rep[sis] * (1.2 if tipo in f_tipo_rep else 1.0) * rng.lognormal(0, 0.35))
            filas_f.append({
                "id_vehiculo": idv,
                "fecha_falla": fecha,
                "sistema": sis,
                "turno": turno,
                "id_tecnico": f"T{int(rng.integers(1, 13)):03d}",
                "horas_reparacion": float(round(min(max(horas_rep, 0.5), 120), 1)),
                "costo_cop": float(round(costo, -3)),
                "km_al_fallar": int(km_v[idv] + rng.integers(0, 60000)),
            })
    fal = pd.DataFrame(filas_f).sort_values(["fecha_falla", "id_vehiculo"]).reset_index(drop=True)
    fal.insert(0, "id_orden", [f"OT-{i + 1:05d}" for i in range(len(fal))])

    # ---------- 4. Problemas de calidad a propósito ----------
    n_f = len(fal)

    def sin_tildes(s):
        return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

    # 4.1 textos con mayúsculas / tildes / espacios mezclados en 'sistema'
    sis_raw = fal["sistema"].astype(object).copy()
    for i in rng.choice(n_f, int(n_f * 0.25), replace=False):
        v = sis_raw[i]
        op = int(rng.integers(0, 4))
        sis_raw[i] = [v.upper(), v.lower(), " " + sin_tildes(v) + " ", sin_tildes(v)][op]
    fal["sistema"] = sis_raw

    # 4.2 identificadores de vehículo mal escritos (minúscula o espacio final)
    idv_raw = fal["id_vehiculo"].astype(object).copy()
    for i in rng.choice(n_f, 20, replace=False):
        idv_raw[i] = idv_raw[i].lower() if rng.random() < 0.5 else idv_raw[i] + " "
    fal["id_vehiculo"] = idv_raw

    # 4.3 fechas en tres formatos
    abr = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
    fechas = []
    for f in fal["fecha_falla"]:
        r = rng.random()
        if r < 0.70:
            fechas.append(f.strftime("%Y-%m-%d"))
        elif r < 0.90:
            fechas.append(f.strftime("%d/%m/%Y"))
        else:
            fechas.append(f"{f.day:02d}-{abr[f.month - 1]}-{f.year}")
    fal["fecha_falla"] = fechas

    # 4.4 valores imposibles y vacíos
    horas_rep_col = fal["horas_reparacion"].astype(float).copy()
    idx = rng.permutation(n_f)
    horas_rep_col.iloc[idx[:15]] = -horas_rep_col.iloc[idx[:15]].abs()      # negativos
    horas_rep_col.iloc[idx[15:23]] = 999.0                                  # exagerados
    horas_rep_col.iloc[idx[23:83]] = np.nan                                 # vacíos
    fal["horas_reparacion"] = horas_rep_col
    costo_col = fal["costo_cop"].astype(float).copy()
    costo_col.iloc[rng.choice(n_f, 40, replace=False)] = np.nan
    fal["costo_cop"] = costo_col

    # 4.5 duplicados exactos (misma orden registrada dos veces)
    dup = fal.sample(45, random_state=int(semilla)).copy()
    fal = pd.concat([fal, dup], ignore_index=True)
    fal = fal.iloc[rng.permutation(len(fal))].reset_index(drop=True)

    # 4.6 tipo de vehículo con textos inconsistentes
    tipo_raw = veh["tipo"].astype(object).copy()
    for i in rng.choice(len(veh), 8, replace=False):
        v = tipo_raw[i]
        tipo_raw[i] = v.upper() if rng.random() < 0.5 else sin_tildes(v).lower()
    veh["tipo"] = tipo_raw

    # 4.7 horas de operación vacías
    hor.loc[rng.choice(len(hor), 12, replace=False), "horas_operadas"] = np.nan

    veh.to_csv(os.path.join(carpeta, "vehiculos.csv"), index=False, encoding="utf-8")
    hor.to_csv(os.path.join(carpeta, "horas_operacion.csv"), index=False, encoding="utf-8")
    fal.to_csv(os.path.join(carpeta, "fallas.csv"), index=False, encoding="utf-8")
    return {"vehiculos": veh.shape, "horas_operacion": hor.shape, "fallas": fal.shape}

**2.2 · Generación y lectura de las fuentes.**

In [37]:
info = generar_datos("datos", semilla=2026)

veh_raw = pd.read_csv("datos/vehiculos.csv")
hor_raw = pd.read_csv("datos/horas_operacion.csv")
fal_raw = pd.read_csv("datos/fallas.csv")
fuentes = {"vehiculos": veh_raw, "horas_operacion": hor_raw, "fallas": fal_raw}

resumen = pd.DataFrame(
    [{"fuente": k, "filas": d.shape[0], "columnas": d.shape[1]} for k, d in fuentes.items()]
)
display(resumen)
for nombre, d in fuentes.items():
    print(f"\n--- {nombre}: {d.shape[0]} filas x {d.shape[1]} columnas ---")
    display(d.head(5))

,fuente,filas,columnas
0,vehiculos,60,6
1,horas_operacion,720,3
2,fallas,1522,9



--- vehiculos: 60 filas x 6 columnas ---


,id_vehiculo,tipo,marca,anio,sede,km_inicial
0,V001,Tractocamión,Kenworth,2021,Pitalito,254820
1,V002,Tractocamión,Kenworth,2015,Neiva,562265
2,V003,Tractocamión,Freightliner,2021,Garzón,308180
3,V004,Tractocamión,Freightliner,2010,Neiva,953328
4,V005,Tractocamión,Freightliner,2010,Garzón,775584



--- horas_operacion: 720 filas x 3 columnas ---


,id_vehiculo,mes,horas_operadas
0,V001,2025-11,429.7
1,V001,2025-12,471.7
2,V001,2026-01,418.1
3,V001,2026-02,365.3
4,V001,2026-03,389.4



--- fallas: 1522 filas x 9 columnas ---


,id_orden,id_vehiculo,fecha_falla,sistema,turno,id_tecnico,horas_reparacion,costo_cop,km_al_fallar
0,OT-00895,V030,2026-06-22,Eléctrico,Mañana,T003,3.0,593000.0,455769
1,OT-01240,V056,2026-09-14,Suspensión,Tarde,T010,9.2,1254000.0,281536
2,OT-00299,V047,2026-01-11,Transmision,Noche,T005,24.8,3613000.0,885407
3,OT-00150,V009,2025-12-10,Transmisión,Noche,T007,30.3,4240000.0,272933
4,OT-00461,V020,2026-03-09,Llantas,Tarde,T012,2.3,870000.0,951191


## 3 · Diagnóstico de calidad

*Qué se hace y por qué:* antes de calcular cualquier KPI se revisa cada fuente en busca de **vacíos, duplicados, formatos de fecha, textos inconsistentes y valores imposibles**. Un MTBF calculado sobre datos sucios (por ejemplo, con órdenes repetidas o vehículos que no cruzan por un identificador mal escrito) daría una cifra equivocada.

In [38]:
def quitar_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")

def normalizar(s):
    return quitar_tildes(s).strip().lower()

def formato_fecha(s):
    s = str(s).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s): return "AAAA-MM-DD"
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s): return "DD/MM/AAAA"
    if re.fullmatch(r"\d{2}-[a-z]{3}-\d{4}", s): return "DD-mmm-AAAA"
    return "otro"

diag = []
def reportar(fuente, problema, columna, filas, total, detalle=""):
    diag.append({"fuente": fuente, "problema": problema, "columna": columna,
                 "filas_afectadas": int(filas), "porcentaje": f"{filas/total:.1%}", "detalle": detalle})

# 1. Vacíos
for nombre, d in fuentes.items():
    for col, n in d.isna().sum().items():
        if n > 0:
            reportar(nombre, "Valores vacíos", col, n, len(d))

# 2. Duplicados exactos
reportar("fallas", "Filas duplicadas", "(fila completa)", fal_raw.duplicated().sum(), len(fal_raw),
         "misma orden de trabajo registrada dos veces")

# 3. Formatos de fecha
formatos = fal_raw["fecha_falla"].map(formato_fecha).value_counts()
reportar("fallas", "Fechas en formatos distintos", "fecha_falla", (formatos.drop("AAAA-MM-DD", errors="ignore")).sum(),
         len(fal_raw), " | ".join(f"{k}: {v}" for k, v in formatos.items()))

# 4. Textos inconsistentes
sis_crudo, sis_norm = fal_raw["sistema"].nunique(), fal_raw["sistema"].map(normalizar).nunique()
reportar("fallas", "Textos inconsistentes", "sistema",
         (fal_raw["sistema"] != fal_raw["sistema"].str.strip().str.title()).sum(), len(fal_raw),
         f"{sis_crudo} variantes escritas para {sis_norm} sistemas reales")
tipo_crudo, tipo_norm = veh_raw["tipo"].nunique(), veh_raw["tipo"].map(normalizar).nunique()
reportar("vehiculos", "Textos inconsistentes", "tipo",
         (veh_raw["tipo"] != veh_raw["tipo"].str.strip().str.title()).sum(), len(veh_raw),
         f"{tipo_crudo} variantes escritas para {tipo_norm} tipos reales")
ids_mal = (fal_raw["id_vehiculo"] != fal_raw["id_vehiculo"].str.strip().str.upper()).sum()
huerfanas = (~fal_raw["id_vehiculo"].isin(veh_raw["id_vehiculo"])).sum()
reportar("fallas", "Identificador mal escrito", "id_vehiculo", ids_mal, len(fal_raw),
         f"{huerfanas} órdenes no cruzarían con el catálogo de vehículos")

# 5. Valores imposibles
neg = (fal_raw["horas_reparacion"] <= 0).sum()
alto = (fal_raw["horas_reparacion"] > 240).sum()
reportar("fallas", "Valores imposibles", "horas_reparacion", neg + alto, len(fal_raw),
         f"{neg} horas negativas o cero y {alto} reparaciones de más de 240 h (10 días)")

diagnostico = pd.DataFrame(diag)
display(diagnostico)

,fuente,problema,columna,filas_afectadas,porcentaje,detalle
0,horas_operacion,Valores vacíos,horas_operadas,12,1.7%,
1,fallas,Valores vacíos,horas_reparacion,61,4.0%,
2,fallas,Valores vacíos,costo_cop,41,2.7%,
3,fallas,Filas duplicadas,(fila completa),45,3.0%,misma orden de trabajo registrada dos veces
4,fallas,Fechas en formatos distintos,fecha_falla,460,30.2%,AAAA-MM-DD: 1062 | DD/MM/AAAA: 307 | DD-mmm-AA...
5,fallas,Textos inconsistentes,sistema,283,18.6%,32 variantes escritas para 7 sistemas reales
6,vehiculos,Textos inconsistentes,tipo,8,13.3%,10 variantes escritas para 5 tipos reales
7,fallas,Identificador mal escrito,id_vehiculo,22,1.4%,22 órdenes no cruzarían con el catálogo de veh...
8,fallas,Valores imposibles,horas_reparacion,26,1.7%,17 horas negativas o cero y 9 reparaciones de ...


**Lectura del diagnóstico.** La tabla principal (`fallas`) concentra los problemas: contiene órdenes duplicadas, fechas escritas de tres maneras, nombres de sistema con mayúsculas y tildes mezcladas, identificadores de vehículo mal escritos (que romperían el cruce con el catálogo), reparaciones con horas negativas o exageradas y vacíos en horas de reparación y costo. Las otras dos fuentes tienen problemas menores: el tipo de vehículo escrito de varias formas y horas de operación vacías.

## 4 · Limpieza

*Qué se hace y por qué:* cada corrección se aplica sobre una **copia** de los datos crudos y se documenta con su **antes**, su **después** y el **motivo** de la decisión (eliminar, imputar o corregir). Al final se presenta una bitácora de cambios.

In [39]:
veh, hor, fal = veh_raw.copy(), hor_raw.copy(), fal_raw.copy()
bitacora = []
def registrar(problema, fuente, columna, filas, decision, regla, motivo):
    bitacora.append({"problema": problema, "fuente": fuente, "columna": columna,
                     "filas_afectadas": int(filas), "decision": decision, "regla": regla, "motivo": motivo})

**4.1 · Duplicados exactos → eliminar.**

In [40]:
antes = len(fal)
ejemplo_id = fal[fal.duplicated(keep=False)].sort_values("id_orden").iloc[0]["id_orden"]
print("ANTES: filas =", antes, "| ejemplo de orden repetida:")
display(fal[fal["id_orden"] == ejemplo_id])

n_dup = int(fal.duplicated().sum())
fal = fal.drop_duplicates().reset_index(drop=True)
print("DESPUÉS: filas =", len(fal), "| duplicados restantes =", int(fal.duplicated().sum()))
assert fal["id_orden"].is_unique
registrar("Filas duplicadas", "fallas", "(fila completa)", n_dup, "Eliminar",
          "drop_duplicates()", "Una orden de trabajo repetida cuenta dos veces la misma falla e infla el conteo y baja el MTBF.")

ANTES: filas = 1522 | ejemplo de orden repetida:


,id_orden,id_vehiculo,fecha_falla,sistema,turno,id_tecnico,horas_reparacion,costo_cop,km_al_fallar
434,OT-00027,V018,07/11/2025,Motor,Mañana,T003,27.2,2618000.0,886345
1424,OT-00027,V018,07/11/2025,Motor,Mañana,T003,27.2,2618000.0,886345


DESPUÉS: filas = 1477 | duplicados restantes = 0


**4.2 · Fechas en tres formatos → corregir (convertir a fecha real).**

In [41]:
MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
         "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12}

def parsear_fecha(s):
    s = str(s).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return pd.Timestamp(s)
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):
        d, m, a = s.split("/")
        return pd.Timestamp(int(a), int(m), int(d))
    d, m, a = s.split("-")
    return pd.Timestamp(int(a), MESES[m.lower()], int(d))

muestra_antes = fal.groupby(fal["fecha_falla"].map(formato_fecha)).head(2)[["id_orden", "fecha_falla"]]
print("ANTES (texto en tres formatos):")
display(muestra_antes.sort_values("fecha_falla"))
n_fechas = int((fal["fecha_falla"].map(formato_fecha) != "AAAA-MM-DD").sum())

fal["fecha"] = fal["fecha_falla"].map(parsear_fecha)
fal["fecha_falla"] = fal["fecha"].dt.strftime("%Y-%m-%d")
print("DESPUÉS: tipo =", fal["fecha"].dtype, "| rango:", fal["fecha"].min().date(), "a", fal["fecha"].max().date())
display(fal.loc[muestra_antes.index, ["id_orden", "fecha_falla"]])
assert fal["fecha_falla"].map(formato_fecha).eq("AAAA-MM-DD").all()
registrar("Fechas en formatos distintos", "fallas", "fecha_falla", n_fechas, "Corregir",
          "Convertir AAAA-MM-DD, DD/MM/AAAA y DD-mmm-AAAA a una fecha única AAAA-MM-DD",
          "Sin una fecha homogénea no se pueden agrupar las fallas por mes ni filtrar por periodos.")

ANTES (texto en tres formatos):


,id_orden,fecha_falla
7,OT-00373,03/02/2026
43,OT-01346,09-oct-2026
21,OT-00994,11-jul-2026
0,OT-00895,2026-06-22
1,OT-01240,2026-09-14
5,OT-01474,31/10/2026


DESPUÉS: tipo = datetime64[ns] | rango: 2025-11-01 a 2026-10-31


,id_orden,fecha_falla
0,OT-00895,2026-06-22
1,OT-01240,2026-09-14
5,OT-01474,2026-10-31
7,OT-00373,2026-02-03
21,OT-00994,2026-07-11
43,OT-01346,2026-10-09


**4.3 · Textos inconsistentes → corregir (normalizar).**

In [42]:
print("ANTES — sistemas distintos escritos:", fal["sistema"].nunique())
print(sorted(fal["sistema"].unique())[:14], "...")

SISTEMAS = {"motor": "Motor", "transmision": "Transmisión", "frenos": "Frenos", "electrico": "Eléctrico",
            "suspension": "Suspensión", "llantas": "Llantas", "refrigeracion": "Refrigeración"}
TIPOS = {"tractocamion": "Tractocamión", "camion": "Camión", "volqueta": "Volqueta",
         "bus": "Bus", "camioneta": "Camioneta"}

n_sis = int((fal["sistema"] != fal["sistema"].map(lambda s: SISTEMAS[normalizar(s)])).sum())
fal["sistema"] = fal["sistema"].map(lambda s: SISTEMAS[normalizar(s)])

n_tipo = int((veh["tipo"] != veh["tipo"].map(lambda s: TIPOS[normalizar(s)])).sum())
print("ANTES — tipos de vehículo distintos escritos:", veh["tipo"].nunique(), sorted(veh["tipo"].unique()))
veh["tipo"] = veh["tipo"].map(lambda s: TIPOS[normalizar(s)])

n_ids = int((fal["id_vehiculo"] != fal["id_vehiculo"].str.strip().str.upper()).sum())
fal["id_vehiculo"] = fal["id_vehiculo"].str.strip().str.upper()

print("\nDESPUÉS — sistemas:", fal["sistema"].nunique(), sorted(fal["sistema"].unique()))
print("DESPUÉS — tipos:", veh["tipo"].nunique(), sorted(veh["tipo"].unique()))
print("DESPUÉS — órdenes sin vehículo en el catálogo:", int((~fal["id_vehiculo"].isin(veh["id_vehiculo"])).sum()))
assert fal["id_vehiculo"].isin(veh["id_vehiculo"]).all()

registrar("Textos inconsistentes", "fallas", "sistema", n_sis, "Corregir",
          "strip + minúsculas + quitar tildes, luego mapa a nombre oficial",
          "Un mismo sistema escrito de varias formas se contaría como sistemas distintos.")
registrar("Textos inconsistentes", "vehiculos", "tipo", n_tipo, "Corregir", "Mismo procedimiento con el mapa de tipos",
          "Evita partir un mismo tipo de vehículo en varios grupos al calcular el MTBF por tipo.")
registrar("Identificador mal escrito", "fallas", "id_vehiculo", n_ids, "Corregir", "strip + mayúsculas",
          "Los espacios y minúsculas impedían cruzar las órdenes con el catálogo de vehículos.")

ANTES — sistemas distintos escritos: 32
[' Electrico ', ' Frenos ', ' Llantas ', ' Motor ', ' Refrigeracion ', ' Suspension ', ' Transmision ', 'ELÉCTRICO', 'Electrico', 'Eléctrico', 'FRENOS', 'Frenos', 'LLANTAS', 'Llantas'] ...
ANTES — tipos de vehículo distintos escritos: 10 ['BUS', 'Bus', 'CAMIONETA', 'CAMIÓN', 'Camioneta', 'Camión', 'Tractocamión', 'VOLQUETA', 'Volqueta', 'camioneta']

DESPUÉS — sistemas: 7 ['Eléctrico', 'Frenos', 'Llantas', 'Motor', 'Refrigeración', 'Suspensión', 'Transmisión']
DESPUÉS — tipos: 5 ['Bus', 'Camioneta', 'Camión', 'Tractocamión', 'Volqueta']
DESPUÉS — órdenes sin vehículo en el catálogo: 0


**4.4 · Valores imposibles en `horas_reparacion` → imputar con la mediana del sistema.** Se consideran imposibles las horas menores o iguales a cero y las mayores a 240 (diez días corridos).

In [43]:
h = fal["horas_reparacion"]
print("ANTES: mínimo =", h.min(), "| máximo =", h.max(), "| promedio =", round(h.mean(), 2), "| vacíos =", int(h.isna().sum()))

imposible = (h <= 0) | (h > 240)
n_imp, n_vac = int(imposible.sum()), int(h.isna().sum())
fal.loc[imposible, "horas_reparacion"] = np.nan
mediana_sis = fal.groupby("sistema")["horas_reparacion"].transform("median")
fal["horas_reparacion"] = fal["horas_reparacion"].fillna(mediana_sis)

h = fal["horas_reparacion"]
print("DESPUÉS: mínimo =", h.min(), "| máximo =", h.max(), "| promedio =", round(h.mean(), 2), "| vacíos =", int(h.isna().sum()))
assert h.between(0.1, 240).all()
registrar("Valores imposibles", "fallas", "horas_reparacion", n_imp, "Imputar",
          "Marcar como vacío si ≤ 0 o > 240 h y reemplazar por la mediana de su sistema",
          "No se puede reparar en tiempo negativo; la mediana por sistema es robusta frente a extremos.")
registrar("Valores vacíos", "fallas", "horas_reparacion", n_vac, "Imputar", "Mediana de la reparación por sistema",
          "Conservar la orden (la falla ocurrió) sin sesgar el MTTR.")

ANTES: mínimo = -39.8 | máximo = 999.0 | promedio = 15.56 | vacíos = 60
DESPUÉS: mínimo = 0.6 | máximo = 83.1 | promedio = 10.18 | vacíos = 0


**4.5 · Horas de operación vacías → imputar con el promedio del vehículo.** Y **4.6 · Costos vacíos → no imputar** (se excluyen del promedio).

In [44]:
n_h = int(hor["horas_operadas"].isna().sum())
print("ANTES: vacíos en horas_operadas =", n_h, "| total de horas =", round(hor["horas_operadas"].sum(), 1))
prom_veh = hor.groupby("id_vehiculo")["horas_operadas"].transform("mean")
hor["horas_operadas"] = hor["horas_operadas"].fillna(prom_veh).round(1)
print("DESPUÉS: vacíos =", int(hor["horas_operadas"].isna().sum()), "| total de horas =", round(hor["horas_operadas"].sum(), 1))
registrar("Valores vacíos", "horas_operacion", "horas_operadas", n_h, "Imputar", "Promedio de horas del mismo vehículo",
          "El MTBF necesita las horas de todos los vehículos; el promedio propio respeta su nivel de uso.")

n_c = int(fal["costo_cop"].isna().sum())
print("\nCostos vacíos:", n_c, "de", len(fal), "→ se conservan vacíos y se excluyen del promedio de costo.")
registrar("Valores vacíos", "fallas", "costo_cop", n_c, "Conservar vacío",
          "No imputar; el promedio ignora los vacíos", "Inventar un costo distorsionaría el costo promedio por falla.")

ANTES: vacíos en horas_operadas = 12 | total de horas = 217285.2
DESPUÉS: vacíos = 0 | total de horas = 220758.0

Costos vacíos: 40 de 1477 → se conservan vacíos y se excluyen del promedio de costo.


**4.7 · Columnas derivadas y bitácora final de cambios.**

In [45]:
fal["mes"] = fal["fecha"].dt.strftime("%Y-%m")
fal["semestre"] = np.where(fal["fecha"] >= pd.Timestamp("2026-05-01"), "S2", "S1")
hor["semestre"] = np.where(hor["mes"] >= "2026-05", "S2", "S1")
SEM = {"S1": "S1 (nov-abr)", "S2": "S2 (may-oct)"}
fal = fal.drop(columns=["fecha"])

bitacora_df = pd.DataFrame(bitacora)
display(bitacora_df)
print(f"Filas de fallas: {len(fal_raw)} → {len(fal)} | horas_operacion: {len(hor_raw)} → {len(hor)} | vehiculos: {len(veh_raw)} → {len(veh)}")

,problema,fuente,columna,filas_afectadas,decision,regla,motivo
0,Filas duplicadas,fallas,(fila completa),45,Eliminar,drop_duplicates(),Una orden de trabajo repetida cuenta dos veces...
1,Fechas en formatos distintos,fallas,fecha_falla,448,Corregir,"Convertir AAAA-MM-DD, DD/MM/AAAA y DD-mmm-AAAA...",Sin una fecha homogénea no se pueden agrupar l...
2,Textos inconsistentes,fallas,sistema,322,Corregir,"strip + minúsculas + quitar tildes, luego mapa...",Un mismo sistema escrito de varias formas se c...
3,Textos inconsistentes,vehiculos,tipo,8,Corregir,Mismo procedimiento con el mapa de tipos,Evita partir un mismo tipo de vehículo en vari...
4,Identificador mal escrito,fallas,id_vehiculo,20,Corregir,strip + mayúsculas,Los espacios y minúsculas impedían cruzar las ...
5,Valores imposibles,fallas,horas_reparacion,23,Imputar,Marcar como vacío si ≤ 0 o > 240 h y reemplaza...,No se puede reparar en tiempo negativo; la med...
6,Valores vacíos,fallas,horas_reparacion,60,Imputar,Mediana de la reparación por sistema,Conservar la orden (la falla ocurrió) sin sesg...
7,Valores vacíos,horas_operacion,horas_operadas,12,Imputar,Promedio de horas del mismo vehículo,El MTBF necesita las horas de todos los vehícu...
8,Valores vacíos,fallas,costo_cop,40,Conservar vacío,No imputar; el promedio ignora los vacíos,Inventar un costo distorsionaría el costo prom...


Filas de fallas: 1522 → 1477 | horas_operacion: 720 → 720 | vehiculos: 60 → 60


## 5 · Base SQLite y consultas

*Qué se hace y por qué:* se documentan los datos (diccionario y grano), se cargan en una base **SQLite** y se responden partes de la pregunta con SQL. Siguiendo las buenas prácticas de la semana 14, las consultas piden **solo las columnas necesarias** y **filtran antes de agrupar**.

**Grano de la tabla principal (`fallas`):** una fila representa **una orden de trabajo correctiva**, es decir, una falla de un vehículo atendida por el taller en una fecha, un sistema y un turno determinados. `horas_operacion` tiene una fila por vehículo y mes; `vehiculos` tiene una fila por vehículo.

In [46]:
diccionario = pd.DataFrame([
    ("fallas", "id_orden", "texto", "Código único de la orden de trabajo correctiva", "OT-00895"),
    ("fallas", "id_vehiculo", "texto", "Vehículo que falló (clave foránea a vehiculos)", "V030"),
    ("fallas", "fecha_falla", "texto AAAA-MM-DD", "Fecha en que ocurrió la falla", "2026-06-22"),
    ("fallas", "sistema", "texto", "Sistema del vehículo que falló", "Frenos"),
    ("fallas", "turno", "texto", "Turno en que se atendió la falla (Mañana, Tarde, Noche)", "Noche"),
    ("fallas", "id_tecnico", "texto", "Código anonimizado del técnico (T001 a T012)", "T003"),
    ("fallas", "horas_reparacion", "decimal", "Horas que tomó la reparación", "3.0"),
    ("fallas", "costo_cop", "decimal", "Costo de la reparación en pesos colombianos", "593000"),
    ("fallas", "km_al_fallar", "entero", "Kilometraje del vehículo al momento de la falla", "455769"),
    ("fallas", "mes", "texto AAAA-MM", "Mes de la falla (derivada)", "2026-06"),
    ("fallas", "semestre", "texto", "S1 = nov. 2025–abr. 2026; S2 = may.–oct. 2026 (derivada)", "S2"),
    ("horas_operacion", "id_vehiculo", "texto", "Vehículo (clave foránea a vehiculos)", "V001"),
    ("horas_operacion", "mes", "texto AAAA-MM", "Mes de operación", "2025-11"),
    ("horas_operacion", "horas_operadas", "decimal", "Horas que el vehículo operó en el mes", "380.4"),
    ("horas_operacion", "semestre", "texto", "S1 o S2 (derivada)", "S1"),
    ("vehiculos", "id_vehiculo", "texto", "Código único del vehículo (clave primaria)", "V001"),
    ("vehiculos", "tipo", "texto", "Tractocamión, Camión, Volqueta, Bus o Camioneta", "Volqueta"),
    ("vehiculos", "marca", "texto", "Marca del vehículo", "Kenworth"),
    ("vehiculos", "anio", "entero", "Año del modelo", "2016"),
    ("vehiculos", "sede", "texto", "Sede de asignación (Neiva, Pitalito, Garzón)", "Neiva"),
    ("vehiculos", "km_inicial", "entero", "Kilometraje acumulado al inicio del periodo", "420000"),
], columns=["tabla", "columna", "tipo", "descripción", "ejemplo"])
display(diccionario)

,tabla,columna,tipo,descripción,ejemplo
0,fallas,id_orden,texto,Código único de la orden de trabajo correctiva,OT-00895
1,fallas,id_vehiculo,texto,Vehículo que falló (clave foránea a vehiculos),V030
2,fallas,fecha_falla,texto AAAA-MM-DD,Fecha en que ocurrió la falla,2026-06-22
3,fallas,sistema,texto,Sistema del vehículo que falló,Frenos
4,fallas,turno,texto,"Turno en que se atendió la falla (Mañana, Tard...",Noche
5,fallas,id_tecnico,texto,Código anonimizado del técnico (T001 a T012),T003
6,fallas,horas_reparacion,decimal,Horas que tomó la reparación,3.0
7,fallas,costo_cop,decimal,Costo de la reparación en pesos colombianos,593000
8,fallas,km_al_fallar,entero,Kilometraje del vehículo al momento de la falla,455769
9,fallas,mes,texto AAAA-MM,Mes de la falla (derivada),2026-06


In [47]:
if os.path.exists("flota.db"):
    os.remove("flota.db")
con = sqlite3.connect("flota.db")
veh.to_sql("vehiculos", con, index=False)
hor.to_sql("horas_operacion", con, index=False)
fal.to_sql("fallas", con, index=False)
con.execute("CREATE INDEX idx_fallas_veh ON fallas(id_vehiculo)")
con.execute("CREATE INDEX idx_fallas_fecha ON fallas(fecha_falla)")
con.commit()
print(pd.read_sql_query("SELECT name AS tabla FROM sqlite_master WHERE type = 'table'", con))

             tabla
0        vehiculos
1  horas_operacion
2           fallas


**Consulta 1 · GROUP BY:** fallas, tiempo de reparación y costo por turno.

In [48]:
q1 = """
SELECT turno,
       COUNT(*)                         AS n_fallas,
       ROUND(AVG(horas_reparacion), 1)  AS mttr_horas,
       ROUND(AVG(costo_cop))            AS costo_prom_cop
FROM fallas
GROUP BY turno
ORDER BY n_fallas DESC
"""
r1 = pd.read_sql_query(q1, con)
r1["pct_fallas"] = (r1["n_fallas"] / r1["n_fallas"].sum()).map("{:.1%}".format)
display(r1)

,turno,n_fallas,mttr_horas,costo_prom_cop,pct_fallas
0,Noche,638,11.1,1625383.0,43.2%
1,Mañana,420,9.5,1566178.0,28.4%
2,Tarde,419,9.5,1652298.0,28.4%


**Interpretación.** El turno de noche concentra la mayor parte de las fallas (43,2 %) y es el que más tarda en repararlas (11,1 h frente a 9,5 h de los turnos de mañana y tarde). Las reparaciones nocturnas son más largas y más frecuentes, lo que apunta a una cobertura de técnicos insuficiente en ese turno.

**Consulta 2 · JOIN de dos tablas:** MTBF por tipo de vehículo (horas de operación ÷ fallas), cruzando `horas_operacion` y `fallas` con `vehiculos`.

In [49]:
q2 = """
WITH horas AS (
    SELECT v.tipo, SUM(h.horas_operadas) AS horas
    FROM horas_operacion h
    JOIN vehiculos v ON v.id_vehiculo = h.id_vehiculo
    GROUP BY v.tipo
),
n AS (
    SELECT v.tipo, COUNT(*) AS n_fallas
    FROM fallas f
    JOIN vehiculos v ON v.id_vehiculo = f.id_vehiculo
    GROUP BY v.tipo
)
SELECT horas.tipo, ROUND(horas.horas) AS horas_operadas, n.n_fallas,
       ROUND(horas.horas / n.n_fallas, 1) AS mtbf_horas
FROM horas
JOIN n ON n.tipo = horas.tipo
ORDER BY mtbf_horas
"""
r2 = pd.read_sql_query(q2, con)
display(r2)

,tipo,horas_operadas,n_fallas,mtbf_horas
0,Volqueta,45531.0,381,119.5
1,Camión,56573.0,399,141.8
2,Tractocamión,62679.0,437,143.4
3,Bus,32970.0,175,188.4
4,Camioneta,23004.0,85,270.6


**Interpretación.** Las volquetas son el tipo de vehículo con menor MTBF (119,5 h): fallan, en promedio, cada 119,5 horas de operación, frente a 270,6 h de las camionetas. Les siguen los camiones (141,8 h) y los tractocamiones (143,4 h). La confiabilidad depende del tipo de vehículo, por lo que un mismo plan de mantenimiento para toda la flota no es adecuado.

**Consulta 3 · Filtro por fechas:** fallas y tiempo de reparación por mes en el segundo semestre (mayo a octubre de 2026).

In [50]:
q3 = """
SELECT mes,
       COUNT(*)                        AS n_fallas,
       ROUND(AVG(horas_reparacion), 1) AS mttr_horas
FROM fallas
WHERE fecha_falla BETWEEN '2026-05-01' AND '2026-10-31'
GROUP BY mes
ORDER BY mes
"""
r3 = pd.read_sql_query(q3, con)
display(r3)

,mes,n_fallas,mttr_horas
0,2026-05,141,9.5
1,2026-06,173,10.7
2,2026-07,121,10.1
3,2026-08,116,9.0
4,2026-09,131,10.0
5,2026-10,165,9.3


**Interpretación.** El segundo semestre mantiene un nivel alto de fallas con dos picos: junio (173 órdenes) y octubre (165). Julio y agosto bajan a 121 y 116 órdenes, de modo que la carga de averías no es uniforme y conviene preparar el taller antes de los meses pico.

**Consulta 4 · JOIN con filtro:** fallas de frenos y llantas en tractocamiones y volquetas, por semestre.

In [51]:
q4 = """
SELECT v.tipo, f.sistema, f.semestre, COUNT(*) AS n_fallas
FROM fallas f
JOIN vehiculos v ON v.id_vehiculo = f.id_vehiculo
WHERE v.tipo IN ('Tractocamión', 'Volqueta')
  AND f.sistema IN ('Frenos', 'Llantas')
GROUP BY v.tipo, f.sistema, f.semestre
ORDER BY v.tipo, f.sistema, f.semestre
"""
r4 = pd.read_sql_query(q4, con)
display(r4)
tot = r4.groupby("semestre")["n_fallas"].sum()
print(f"Frenos + llantas en tractocamiones y volquetas: S1 = {tot['S1']} → S2 = {tot['S2']} ({tot['S2']/tot['S1']-1:+.1%})")

,tipo,sistema,semestre,n_fallas
0,Tractocamión,Frenos,S1,33
1,Tractocamión,Frenos,S2,58
2,Tractocamión,Llantas,S1,26
3,Tractocamión,Llantas,S2,38
4,Volqueta,Frenos,S1,35
5,Volqueta,Frenos,S2,65
6,Volqueta,Llantas,S1,18
7,Volqueta,Llantas,S2,43


Frenos + llantas en tractocamiones y volquetas: S1 = 112 → S2 = 204 (+82.1%)
